# ODE Inversion & Image Editing with Flow Matching

Since OT-CFM defines a deterministic ODE $dx/dt = v_\theta(x, t)$ from noise ($t{=}0$) to data ($t{=}1$), we can **invert** it by solving backwards ($t{=}1 \to t{=}0$) to recover the noise vector $z_0$ corresponding to any real image $x_1$. We then perform controlled image editing by **perturbing** $z_0$ and re-generating.

**Experiments:**
1. Encode real CIFAR-10 images to noise via ODE inversion
2. Verify reconstruction quality (invert then regenerate)
3. Latent space arithmetic (perturbations in noise space)
4. Interpolation between real images via their latent codes
5. Attribute manipulation by direction vectors

## Setup

In [ ]:
import os, math, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from tqdm import tqdm

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

try:
    from torchdiffeq import odeint
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'torchdiffeq'])
    from torchdiffeq import odeint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [2]:
# Paths
DRIVE_DIR = '/content/drive/MyDrive/flow-matching'
CFM_CKPT  = f'{DRIVE_DIR}/runs/cfm/checkpoints/last.pt'
SAVE_DIR  = f'{DRIVE_DIR}/visualizations/ode_inversion'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Checkpoint: {CFM_CKPT}')
print(f'Save dir:   {SAVE_DIR}')

Checkpoint: /content/drive/MyDrive/flow-matching/runs/cfm/checkpoints/last.pt
Save dir:   /content/drive/MyDrive/flow-matching/visualizations/ode_inversion


## Model Definition

In [3]:
@dataclass
class CFMConfig:
    image_size: int = 32
    channels: int = 3
    model_channels: int = 128
    num_heads: int = 4
    dropout: float = 0.1
    num_groups: int = 32
    sigma_min: float = 0.0
    batch_size: int = 128
    lr: float = 2e-4
    num_epochs: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    clip_denoised: bool = True

cfg = CFMConfig()

In [4]:
class TimeEmbedding(nn.Module):
    def __init__(self, base_dim):
        super().__init__()
        self.base_dim = base_dim
        self.mlp = nn.Sequential(nn.Linear(base_dim, base_dim*4), nn.SiLU(), nn.Linear(base_dim*4, base_dim*4))
    def forward(self, t):
        half  = self.base_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb   = t[:, None].float() * freqs[None]
        emb   = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=32, dropout=0.1):
        super().__init__()
        self.norm1    = nn.GroupNorm(num_groups, in_ch)
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch * 2))
        self.norm2    = nn.GroupNorm(num_groups, out_ch)
        self.dropout  = nn.Dropout(dropout)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act      = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        scale, shift = self.time_mlp(t_emb).chunk(2, dim=-1)
        h = h * (scale[:, :, None, None] + 1) + shift[:, :, None, None]
        h = self.dropout(self.conv2(self.act(self.norm2(h))))
        return h + self.shortcut(x)

class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4, num_groups=32):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.norm      = nn.GroupNorm(num_groups, channels)
        self.to_qkv    = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.to_out    = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.to_qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W).permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.scaled_dot_product_attention(q, k, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.to_out(attn)

class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, t_dim, ng, nh, do = cfg.model_channels, cfg.model_channels*4, cfg.num_groups, cfg.num_heads, cfg.dropout
        self.time_emb = TimeEmbedding(ch)
        self.init_conv = nn.Conv2d(3, ch, 3, padding=1)
        self.enc1_a = ResBlock(ch,   ch,   t_dim, ng, do); self.enc1_b = ResBlock(ch,   ch,   t_dim, ng, do); self.down1 = Downsample(ch)
        self.enc2_a = ResBlock(ch,   ch*2, t_dim, ng, do); self.enc2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn2 = SelfAttention(ch*2, nh, ng); self.down2 = Downsample(ch*2)
        self.enc3_a = ResBlock(ch*2, ch*2, t_dim, ng, do); self.enc3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn3 = SelfAttention(ch*2, nh, ng); self.down3 = Downsample(ch*2)
        self.mid1 = ResBlock(ch*2, ch*2, t_dim, ng, do); self.mid_attn = SelfAttention(ch*2, nh, ng); self.mid2 = ResBlock(ch*2, ch*2, t_dim, ng, do)
        self.up3 = Upsample(ch*2); self.dec3_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn3 = SelfAttention(ch*2, nh, ng)
        self.up2 = Upsample(ch*2); self.dec2_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn2 = SelfAttention(ch*2, nh, ng)
        self.up1 = Upsample(ch*2); self.dec1_a = ResBlock(ch*3, ch,   t_dim, ng, do); self.dec1_b = ResBlock(ch,   ch,   t_dim, ng, do)
        self.out_norm = nn.GroupNorm(ng, ch); self.out_conv = nn.Conv2d(ch, 3, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h = self.init_conv(x)
        h = self.enc1_b(self.enc1_a(h, t_emb), t_emb);  s1 = h
        h = self.attn2(self.enc2_b(self.enc2_a(self.down1(h), t_emb), t_emb)); s2 = h
        h = self.attn3(self.enc3_b(self.enc3_a(self.down2(h), t_emb), t_emb)); s3 = h
        h = self.down3(h); h = self.mid1(h, t_emb); h = self.mid_attn(h); h = self.mid2(h, t_emb)
        h = self.dattn3(self.dec3_b(self.dec3_a(torch.cat([self.up3(h), s3], dim=1), t_emb), t_emb))
        h = self.dattn2(self.dec2_b(self.dec2_a(torch.cat([self.up2(h), s2], dim=1), t_emb), t_emb))
        h = self.dec1_b(self.dec1_a(torch.cat([self.up1(h), s1], dim=1), t_emb), t_emb)
        return self.out_conv(F.silu(self.out_norm(h)))

## Load Model

In [5]:
def load_model(checkpoint_path, use_ema=True):
    assert os.path.exists(checkpoint_path), f'Checkpoint not found: {checkpoint_path}'
    ckpt  = torch.load(checkpoint_path, map_location=device)
    model = UNet(cfg).to(device)
    if use_ema and 'ema_shadow' in ckpt:
        for name, param in model.named_parameters():
            param.data.copy_(ckpt['ema_shadow'][name])
        print(f'Loaded EMA weights (epoch {ckpt["epoch"]}, loss={ckpt["loss"]:.4f})')
    else:
        model.load_state_dict(ckpt['model'])
        print(f'Loaded model weights (epoch {ckpt["epoch"]})')
    model.eval()
    return model

model = load_model(CFM_CKPT)

AssertionError: Checkpoint not found: /content/drive/MyDrive/flow-matching/runs/cfm/checkpoints/last.pt

## Load CIFAR-10 Test Images

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # scale to [-1, 1]
])

cifar_test = datasets.CIFAR10(root='/content/data', train=False, download=True, transform=transform)
class_names = cifar_test.classes
print(f'Loaded {len(cifar_test)} test images, classes: {class_names}')

## Core Functions: Forward ODE & Inverse ODE

In [ ]:
@torch.no_grad()
def ode_forward(model, z0, method='midpoint', nfe=100):
    """Generate images: solve ODE from t=0 (noise) to t=1 (data)."""
    def ode_fn(t, x):
        t_batch = torch.full((x.shape[0],), t.item(), device=device)
        return model(x, t_batch)

    t_span = torch.tensor([0.0, 1.0], device=device)
    nfe_per_step = {'euler': 1, 'midpoint': 2, 'rk4': 4}[method]
    step_size = nfe_per_step / nfe

    x1 = odeint(ode_fn, z0, t_span, method=method,
                options={'step_size': step_size})[-1]
    return x1.clamp(-1, 1)


@torch.no_grad()
def ode_inverse(model, x1, method='midpoint', nfe=100):
    """Encode images to noise: solve ODE from t=1 (data) to t=0 (noise)."""
    def ode_fn(t, x):
        t_batch = torch.full((x.shape[0],), t.item(), device=device)
        return model(x, t_batch)

    t_span = torch.tensor([1.0, 0.0], device=device)
    nfe_per_step = {'euler': 1, 'midpoint': 2, 'rk4': 4}[method]
    step_size = nfe_per_step / nfe

    z0 = odeint(ode_fn, x1, t_span, method=method,
                options={'step_size': step_size})[-1]
    return z0


def to_img(tensor):
    """Convert [-1,1] tensor to displayable image."""
    return (tensor.clamp(-1, 1) * 0.5 + 0.5).cpu()


def show_images(images, titles=None, nrow=None, figsize=None, save_path=None):
    """Display a list of image tensors."""
    n = len(images)
    if nrow is None:
        nrow = n
    if figsize is None:
        figsize = (2.5 * nrow, 2.5 * max(1, (n + nrow - 1) // nrow))
    fig, axes = plt.subplots(max(1, (n + nrow - 1) // nrow), nrow, figsize=figsize)
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = to_img(images[i])
            ax.imshow(img.permute(1, 2, 0).numpy())
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()

## Experiment 1: Reconstruction Quality

Encode real images to noise ($x_1 \to z_0$), then decode back ($z_0 \to \hat{x}_1$). If the ODE is well-learned, reconstruction should be near-perfect.

In [ ]:
# Select diverse test images (one per class)
selected = []
seen_classes = set()
for img, label in cifar_test:
    if label not in seen_classes:
        selected.append((img, label))
        seen_classes.add(label)
    if len(seen_classes) == 10:
        break

images = torch.stack([s[0] for s in selected]).to(device)
labels = [class_names[s[1]] for s in selected]
print(f'Selected {len(images)} images: {labels}')

In [ ]:
# Encode to noise
z0 = ode_inverse(model, images, method='midpoint', nfe=100)
print(f'Latent z0 — mean: {z0.mean():.4f}, std: {z0.std():.4f}')

# Decode back
reconstructed = ode_forward(model, z0, method='midpoint', nfe=100)

# Compute per-image MSE
mse_per_img = ((images - reconstructed) ** 2).mean(dim=[1,2,3])
print(f'Reconstruction MSE per image: {mse_per_img.cpu().numpy().round(5)}')
print(f'Mean reconstruction MSE: {mse_per_img.mean():.6f}')

In [ ]:
# Visualize: Original | Reconstructed | Difference
n = 8
fig, axes = plt.subplots(3, n, figsize=(2.5*n, 7.5))
for i in range(n):
    axes[0, i].imshow(to_img(images[i]).permute(1,2,0).numpy())
    axes[0, i].set_title(labels[i], fontsize=9)
    axes[1, i].imshow(to_img(reconstructed[i]).permute(1,2,0).numpy())
    diff = ((images[i] - reconstructed[i]).abs() * 5).clamp(0, 1)  # amplified
    axes[2, i].imshow(diff.cpu().permute(1,2,0).numpy())

axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
axes[2, 0].set_ylabel('Diff (5x)', fontsize=11)
for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('ODE Inversion: Encode → Decode Reconstruction', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/reconstruction.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/reconstruction.png')
plt.show()

## Experiment 2: Latent Space Perturbations

Perturb $z_0$ with Gaussian noise at various scales and observe the effect on the generated image. Small perturbations should produce subtle variations; large perturbations should change the image significantly.

In [ ]:
# Pick 4 images for perturbation experiments
src_images = images[:4]
src_labels = labels[:4]

# Encode
z0_src = ode_inverse(model, src_images, method='midpoint', nfe=100)

# Perturbation scales
scales = [0.0, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5]
torch.manual_seed(42)

fig, axes = plt.subplots(len(src_images), len(scales), figsize=(2.5*len(scales), 2.5*len(src_images)))

for i in range(len(src_images)):
    noise = torch.randn_like(z0_src[i:i+1])
    for j, s in enumerate(scales):
        z_perturbed = z0_src[i:i+1] + s * noise
        edited = ode_forward(model, z_perturbed, method='midpoint', nfe=100)
        axes[i, j].imshow(to_img(edited[0]).permute(1,2,0).numpy())
        if i == 0:
            axes[i, j].set_title(f'σ={s}', fontsize=10)
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(src_labels[i], fontsize=10)

plt.suptitle('Latent Perturbation: z₀ + σ·ε → x₁', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/perturbations.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/perturbations.png')
plt.show()

## Experiment 3: Interpolation Between Real Images

Given two real images $x_1^A$ and $x_1^B$, encode both to noise: $z_0^A, z_0^B$. Then interpolate in latent space: $z_0^{\alpha} = (1{-}\alpha)\,z_0^A + \alpha\,z_0^B$ and decode each.

In [ ]:
# Select 4 image pairs for interpolation
pairs = [(0, 1), (2, 3), (4, 5), (6, 7)]
alphas = np.linspace(0, 1, 8)

fig, axes = plt.subplots(len(pairs), len(alphas), figsize=(2.2*len(alphas), 2.5*len(pairs)))

z0_all = ode_inverse(model, images, method='midpoint', nfe=100)

for row, (i, j) in enumerate(pairs):
    z_interps = []
    for alpha in alphas:
        z_interps.append((1 - alpha) * z0_all[i] + alpha * z0_all[j])
    z_batch = torch.stack(z_interps)
    decoded = ode_forward(model, z_batch, method='midpoint', nfe=100)

    for col in range(len(alphas)):
        axes[row, col].imshow(to_img(decoded[col]).permute(1,2,0).numpy())
        if row == 0:
            axes[row, col].set_title(f'α={alphas[col]:.2f}', fontsize=8)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{labels[i]}→{labels[j]}', fontsize=8)

plt.suptitle('Interpolation Between Real Images via ODE Inversion', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/real_interpolation.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/real_interpolation.png')
plt.show()

## Experiment 4: Attribute Direction Editing

Compute a mean latent direction between two classes (e.g., car vs. truck) and apply it to edit other images. This is latent arithmetic: $z_0^{\text{edit}} = z_0 + \lambda \cdot (\bar{z}_0^{\text{target}} - \bar{z}_0^{\text{source}})$.

In [ ]:
# Compute class-mean latents from test set
n_per_class = 50
class_images = {c: [] for c in range(10)}

for img, label in cifar_test:
    if len(class_images[label]) < n_per_class:
        class_images[label].append(img)
    if all(len(v) >= n_per_class for v in class_images.values()):
        break

class_mean_z = {}
for c in tqdm(range(10), desc='Computing class-mean latents'):
    batch = torch.stack(class_images[c]).to(device)
    z = ode_inverse(model, batch, method='midpoint', nfe=100)
    class_mean_z[c] = z.mean(dim=0)
    print(f'  {class_names[c]}: z0 mean={class_mean_z[c].mean():.4f}, std={class_mean_z[c].std():.4f}')

In [ ]:
# Direction editing: apply class direction vectors to source images
# Example: take a horse image and push it toward "car" or "ship"
editing_configs = [
    {'source_class': 7, 'target_class': 1, 'name': 'horse → car'},      # horse → automobile
    {'source_class': 3, 'target_class': 5, 'name': 'cat → dog'},         # cat → dog
    {'source_class': 0, 'target_class': 9, 'name': 'airplane → truck'},  # airplane → truck
    {'source_class': 4, 'target_class': 8, 'name': 'deer → ship'},       # deer → ship
]

lambdas = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]

fig, axes = plt.subplots(len(editing_configs), len(lambdas),
                         figsize=(2.2*len(lambdas), 2.5*len(editing_configs)))

for row, ec in enumerate(editing_configs):
    src_c, tgt_c = ec['source_class'], ec['target_class']
    direction = class_mean_z[tgt_c] - class_mean_z[src_c]

    # Take first image of source class
    src_img = class_images[src_c][0].unsqueeze(0).to(device)
    z0_src = ode_inverse(model, src_img, method='midpoint', nfe=100)

    z_edits = []
    for lam in lambdas:
        z_edits.append(z0_src[0] + lam * direction)
    z_batch = torch.stack(z_edits)
    decoded = ode_forward(model, z_batch, method='midpoint', nfe=100)

    for col in range(len(lambdas)):
        axes[row, col].imshow(to_img(decoded[col]).permute(1,2,0).numpy())
        if row == 0:
            axes[row, col].set_title(f'λ={lambdas[col]}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(ec['name'], fontsize=9)

plt.suptitle('Attribute Editing via Latent Direction Vectors', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/attribute_editing.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/attribute_editing.png')
plt.show()

## Experiment 5: Reconstruction vs NFE

How many NFE are needed for accurate inversion? Test reconstruction quality across different NFE budgets.

In [ ]:
test_img = images[:8]
nfe_values = [10, 20, 50, 100, 200]

results = []
for nfe in nfe_values:
    z0 = ode_inverse(model, test_img, method='midpoint', nfe=nfe)
    recon = ode_forward(model, z0, method='midpoint', nfe=nfe)
    mse = ((test_img - recon) ** 2).mean().item()
    results.append({'nfe': nfe, 'mse': mse})
    print(f'NFE={nfe:>4d}  MSE={mse:.6f}')

# Plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r['nfe'] for r in results], [r['mse'] for r in results], 'o-', color='#2962FF', linewidth=2)
ax.set_xlabel('NFE (per direction)')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction Quality vs NFE Budget')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/reconstruction_vs_nfe.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/reconstruction_vs_nfe.png')
plt.show()

## Experiment 6: Inversion Trajectory Visualization

Visualize intermediate states along the ODE inversion path: image → noise.

In [ ]:
@torch.no_grad()
def ode_inverse_trajectory(model, x1, method='midpoint', nfe=100, n_snapshots=8):
    """Encode image to noise, returning intermediate states."""
    def ode_fn(t, x):
        t_batch = torch.full((x.shape[0],), t.item(), device=device)
        return model(x, t_batch)

    t_eval = torch.linspace(1.0, 0.0, n_snapshots, device=device)
    nfe_per_step = {'euler': 1, 'midpoint': 2, 'rk4': 4}[method]
    step_size = nfe_per_step / nfe

    trajectory = odeint(ode_fn, x1, t_eval, method=method,
                        options={'step_size': step_size})
    return trajectory, t_eval

In [ ]:
n_imgs = 4
n_steps = 8
traj, t_vals = ode_inverse_trajectory(model, images[:n_imgs], method='midpoint', nfe=100, n_snapshots=n_steps)

fig, axes = plt.subplots(n_imgs, n_steps, figsize=(2.2*n_steps, 2.5*n_imgs))
for i in range(n_imgs):
    for j in range(n_steps):
        axes[i, j].imshow(to_img(traj[j, i]).permute(1,2,0).numpy())
        if i == 0:
            axes[i, j].set_title(f't={t_vals[j]:.2f}', fontsize=9)
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(labels[i], fontsize=10)

plt.suptitle('ODE Inversion Trajectory: Image → Noise (t=1 → t=0)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/inversion_trajectory.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR}/inversion_trajectory.png')
plt.show()

## Summary

All results saved to `{SAVE_DIR}/`:
- `reconstruction.png` — encode-decode fidelity
- `perturbations.png` — latent noise perturbation effects
- `real_interpolation.png` — interpolation between real images
- `attribute_editing.png` — class direction editing
- `reconstruction_vs_nfe.png` — NFE vs reconstruction quality
- `inversion_trajectory.png` — inversion path visualization